# Universal GCG Attack Demo

> **Note:** This notebook is primarily for demonstration purposes. While it uses the same mechanics as the main attack framework, it relies on a simplified demo config object for illustration. For real experiments and robust results, we strongly advise running the attack using the cli expample notebook or the shell script and config in the `experiments` folder of the `llm` directory.

This notebook demonstrates a minimalistic version of the Universal Greedy Coordinate Gradient (GCG) attack that works across different HuggingFace models. The attack finds adversarial suffixes that can bypass safety filters in language models.

## Key Features
- **Universal**: Works with any HuggingFace model
- **Automatic**: Uses FastChat for conversation template detection
- **Efficient**: Optimized tokenization and processing

In [ ]:
import time
import importlib
import numpy as np
import torch.multiprocessing as mp
import subprocess
import sys
from pathlib import Path
import json

from advsecurenet.llm.GCG.src.conversation.template_utils import get_goals_and_targets, get_workers

print("Imports complete!")

In [ ]:
mp.set_start_method('spawn', force=True)

def dynamic_import(module):
    return importlib.import_module(module)

class SimpleConfig:
    def __init__(self):
        self.attack = "gcg"  
        self.model_name = "gpt2"  # Small Qwen model
        
        self.model_paths = (self.model_name,)
        self.tokenizer_paths = (self.model_name,)
        self.devices = ("cpu",)
        self.device = "cpu"
        self.num_train_models = 1
        self.model_kwargs = [{"low_cpu_mem_usage": True, "use_cache": False}]
        self.tokenizer_kwargs = [{"use_fast": False}]
        self.conversation_templates = (self.model_name,)
        
        # Attack parameters
        self.n_steps = 2
        self.batch_size = 16
        self.topk = 64
        self.temp = 1.0
        self.target_weight = 1.0
        self.control_weight = 0.0
        self.test_steps = 3
        self.anneal = False
        self.incr_control = True
        self.stop_on_success = False
        self.verbose = True
        self.filter_cand = True
        self.allow_non_ascii = False
        self.control_init = "! ! ! ! ! ! ! ! !"
        self.transfer = False
        self.gbda_deterministic = True
        self.lr = 0.001
        
        # Data parameters 
        self.train_data = "harmful_behaviors.csv"
        self.test_data = ""
        self.data_offset = 0
        self.n_train_data = 1
        self.n_test_data = 0
        self.result_prefix = "qwen_attack"

params = SimpleConfig()
print(f"Config created: {params.attack}")
print(f"Model: {params.model_name}")
print(f"Steps: {params.n_steps}, LR: {params.lr}")
print(f"Control init: '{params.control_init}'")
print(f"Batch size: {params.batch_size}, TopK: {params.topk}")

In [ ]:
attack_lib = dynamic_import(f'advsecurenet.llm.GCG.src.{params.attack}')

print("Loading goals and targets...")
train_goals, train_targets, test_goals, test_targets = get_goals_and_targets(params)

# Apply target processing
process_fn = lambda s: s.replace('Sure, h', 'H')
process_fn2 = lambda s: s.replace("Sure, here is", "Sure, here's")
train_targets = [process_fn(t) if np.random.random() < 0.5 else process_fn2(t) for t in train_targets]
test_targets = [process_fn(t) if np.random.random() < 0.5 else process_fn2(t) for t in test_targets]

print("Loading workers...")
workers, test_workers = get_workers(params)

print(f"Loaded:")
print(f"   Train goals: {len(train_goals)}")
print(f"   Train targets: {len(train_targets)}")
print(f"   Workers: {len(workers)}")
print(f"   Goal: {train_goals[0][:50]}...")
print(f"   Target: {train_targets[0]}")

In [ ]:
import os

print(f"Current working directory: {os.getcwd()}")
print(f"Files in current directory: {os.listdir('.')}")

# Check if file exists
if os.path.exists("harmful_behaviors.csv"):
    print("✓ harmful_behaviors.csv found in current directory")
elif os.path.exists("../harmful_behaviors.csv"):
    print("✓ harmful_behaviors.csv found in parent directory")
else:
    print("✗ harmful_behaviors.csv not found")

In [ ]:
managers = {
    "AP": attack_lib.AttackPrompt,
    "PM": attack_lib.PromptManager,
    "MPA": attack_lib.MultiPromptAttack,
}

timestamp = time.strftime("%Y%m%d-%H:%M:%S")
model_name_clean = params.model_name.replace('/', '_').replace('-', '_')
logfile = f"{model_name_clean}_attack_{timestamp}.json"

attack = attack_lib.IndividualPromptAttack(
    train_goals,
    train_targets,
    workers,
    control_init=params.control_init,
    logfile=logfile,
    managers=managers,
    test_goals=getattr(params, 'test_goals', []),
    test_targets=getattr(params, 'test_targets', []),
    test_workers=test_workers,
    mpa_deterministic=params.gbda_deterministic,
    mpa_lr=params.lr,
    mpa_batch_size=params.batch_size,
    mpa_n_steps=params.n_steps,
)

print(f"Attack created: {type(attack).__name__}")
print(f"Using IndividualPromptAttack with control: '{params.control_init}'")

# Single attack run

In [ ]:
print("Running attack...")
print("=" * 50)

attack.run(
    n_steps=params.n_steps,
    batch_size=params.batch_size, 
    topk=params.topk,
    temp=params.temp,
    target_weight=params.target_weight,
    control_weight=params.control_weight,
    test_steps=getattr(params, 'test_steps', 1),
    anneal=params.anneal,
    incr_control=params.incr_control,
    stop_on_success=params.stop_on_success,
    verbose=params.verbose,
    filter_cand=params.filter_cand,
    allow_non_ascii=params.allow_non_ascii,
)

print("=" * 50)
print("Attack completed!")
print("Cleaning up workers...")

for worker in workers + test_workers:
    worker.stop()

print("Cleanup complete!")

with open(logfile, 'r') as f:
    results = json.load(f)
    

print(f"Results saved to: {logfile}")
final_suffix = results['controls'][-1]
print(f"Final adversarial suffix: '{repr(final_suffix)}'")
print(f"Final loss: {results['losses'][-1]}")
print(f"Original goal: '{train_goals[0]}'")
print(f"Target response: '{train_targets[0]}'")
print(f"Model tested: {params.model_name}")


## Multi-Model GCG Attack Experiment

The following experiment tests the GCG attack across three different language models to evaluate the universality and effectiveness of the adversarial suffix generation approach.

### Models Under Test

1. **Qwen/Qwen2-0.5B** - Alibaba's lightweight conversational model
2. **microsoft/DialoGPT-small** - Microsoft's dialog-optimized GPT model  
3. **TinyLlama/TinyLlama-1.1B-Chat-v1.0** - Compact chat-tuned Llama variant

### Experiment Parameters

- **Attack Steps**: 5 optimization iterations per model
- **Target Goal**: Generate script for exploiting software vulnerabilities
- **Control Initialization**: "! ! ! ! ! ! ! ! !"
- **Batch Size**: 16 candidates per iteration
- **Evaluation**: Individual prompt attack (non-transfer)

### Expected Outcomes

Each model will generate an optimized adversarial suffix designed to bypass safety filters and elicit the target harmful response. Results will show model-specific suffix variations and optimization effectiveness.

In [ ]:


models_to_test = [
    {"name": "Qwen/Qwen2-0.5B", "steps": 5},
    {"name": "microsoft/DialoGPT-small", "steps": 5},
    {"name": "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "steps": 5}
]

results_summary = []

for model_config in models_to_test:
    model_name = model_config["name"]
    n_steps = model_config["steps"]
    
    print(f"\n{'='*60}")
    print(f"Testing model: {model_name}")
    print(f"Steps: {n_steps}")
    print(f"{'='*60}")
    
    try:
        params.model_name = model_name
        params.n_steps = n_steps
        params.model_paths = (model_name,)
        params.tokenizer_paths = (model_name,)
        params.conversation_templates = (model_name,)
        
        print("Loading workers...")
        workers, test_workers = get_workers(params)
        
        timestamp = time.strftime("%Y%m%d-%H:%M:%S")
        model_name_clean = model_name.replace('/', '_').replace('-', '_')
        logfile = f"{model_name_clean}_attack_{timestamp}.json"
        
        attack = attack_lib.IndividualPromptAttack(
            train_goals,
            train_targets,
            workers,
            control_init=params.control_init,
            logfile=logfile,
            managers=managers,
            test_goals=[],
            test_targets=[],
            test_workers=test_workers,
            mpa_deterministic=params.gbda_deterministic,
            mpa_lr=params.lr,
            mpa_batch_size=params.batch_size,
            mpa_n_steps=params.n_steps,
        )
        
        print("Running attack...")
        attack.run(
            n_steps=params.n_steps,
            batch_size=params.batch_size,
            topk=params.topk,
            temp=params.temp,
            target_weight=params.target_weight,
            control_weight=params.control_weight,
            test_steps=params.test_steps,
            anneal=params.anneal,
            incr_control=params.incr_control,
            stop_on_success=params.stop_on_success,
            verbose=params.verbose,
            filter_cand=params.filter_cand,
            allow_non_ascii=params.allow_non_ascii,
        )
        
        for worker in workers + test_workers:
            worker.stop()
        
        with open(logfile, 'r') as f:
            results = json.load(f)
        
        final_suffix = results['controls'][-1] if results['controls'] else "None"
        final_loss = results['losses'][-1] if results['losses'] else "N/A"
        
        results_summary.append({
            "model": model_name,
            "suffix": final_suffix,
            "loss": final_loss,
            "status": "SUCCESS"
        })
        
        print(f"Status: SUCCESS")
        print(f"Final suffix: '{repr(final_suffix)}'")
        print(f"Final loss: {final_loss}")
        
    except Exception as e:
        print(f"Status: FAILED - {e}")
        results_summary.append({
            "model": model_name,
            "suffix": "FAILED",
            "loss": "N/A",
            "status": "FAILED"
        })

print(f"\n{'='*80}")
print("EXPERIMENT SUMMARY")
print(f"{'='*80}")

for result in results_summary:
    print(f"Model: {result['model']}")
    print(f"Status: {result['status']}")
    print(f"Final Loss: {result['loss']}")
    print(f"Final Suffix: '{result['suffix']}'")
    print("-" * 40)

print(f"Total models tested: {len(results_summary)}")
successful = sum(1 for r in results_summary if r['status'] == 'SUCCESS')
print(f"Successful attacks: {successful}/{len(results_summary)}")